# EDA de retrasos de vuelos — notebook **reconstruido desde la conversación**

⚠️ **NO es el `.ipynb` original del equipo.** Se recuperó el diseño y las transformaciones esenciales discutidas en el chat para poder abrirlos y editarlos en Google Colab. No contiene resultados ejecutados ni modelos entrenados.  

**Datos esperados:** `muestra_historica_bts.parquet` (muestra mensual de BTS), el archivo generado en Colab. La muestra NO representa el pipeline final de Databricks.  
**Fuentes:** [BTS](https://transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ) / [PREZIP](https://transtats.bts.gov/PREZIP/).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 100)
print('EDA reconstruido; sin datos descargados incorporados.')

## 1. Cargar muestra guardada
Se espera el Parquet del EDA previo. Puedes subirlo al entorno Colab o copiarlo a Google Drive. El código no intentará descargar datasets completos automáticamente.

In [ ]:
CANDIDATOS = [
  Path('/content/proyecto_retrasos/raw/muestra_historica_bts.parquet'),
  Path('/content/muestra_historica_bts.parquet')
]
ruta = next((x for x in CANDIDATOS if x.exists()), None)
if ruta is None:
    raise FileNotFoundError('Sube muestra_historica_bts.parquet a /content o ajusta CANDIDATOS a su ubicación real.')
historico = pd.read_parquet(ruta)
print(f'Archivo: {ruta}; filas: {len(historico):,}; columnas: {len(historico.columns)}')
display(historico.head())

## 2. Diagnóstico de calidad

In [ ]:
calidad = pd.DataFrame({
  'dtype': historico.dtypes.astype(str),
  'nulos': historico.isna().sum(),
  'nulos_pct': (historico.isna().mean() * 100).round(2)
}).sort_values('nulos_pct', ascending=False)
display(calidad)
print('Duplicados completos:', int(historico.duplicated().sum()))
for c in ['Cancelled', 'Diverted']:
    if c in historico: print(c, historico[c].value_counts(dropna=False).to_dict())

## 3. Preparación del objetivo
La etiqueta es retraso de **salida de puerta** >=15 minutos, con vuelos no cancelados y `DepDelay` disponible. Retener originales para otras investigaciones.

In [ ]:
obligatorias = ['FlightDate','Cancelled','DepDelay','Origin','Reporting_Airline','CRSDepTime']
faltantes = [c for c in obligatorias if c not in historico]
if faltantes: raise ValueError(f'Faltan columnas BTS: {faltantes}')
df = historico.copy()
df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')
for c in ['Cancelled','DepDelay','CRSDepTime']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
eda = df.loc[(df['Cancelled']==0)&df['DepDelay'].notna()&df['FlightDate'].notna()].copy()
eda['delay_15']=(eda['DepDelay']>=15).astype('int8')
eda['year']=eda['FlightDate'].dt.year
eda['month']=eda['FlightDate'].dt.month
eda['day_of_week']=eda['FlightDate'].dt.dayofweek
eda['scheduled_hour']=(eda['CRSDepTime']//100).where(lambda s:s.between(0,23))
eda['positive_delay_min']=eda['DepDelay'].clip(lower=0)
print('Operados validos:',len(eda)); print('Retrasos >=15:',int(eda['delay_15'].sum())); print('Tasa:',eda['delay_15'].mean())

## 4. Distribución de retrasos (EDA preliminar)

In [ ]:
retrasos=eda['DepDelay'].dropna()
positivos=retrasos[retrasos>0]
fig,ax=plt.subplots(1,2,figsize=(14,5))
sns.histplot(retrasos,bins=100,ax=ax[0]); ax[0].set_xlim(-40,180);ax[0].axvline(15,ls='--',color='red');ax[0].set_title('Todos los retrasos (min)')
sns.histplot(positivos,bins=80,ax=ax[1],stat='density');ax[1].set_xlim(0,180);ax[1].set_title('Retrasos positivos (min)')
plt.tight_layout();plt.show()
print('Media todos:',retrasos.mean(),'Mediana todos:',retrasos.median())
display(positivos.quantile([.50,.75,.90,.95,.99]).to_frame('Minutos'))

## 5. Estacionalidad por año y mes

In [ ]:
mensual=eda.groupby(['year','month']).agg(vuelos=('delay_15','size'), tasa=('delay_15','mean')).reset_index()
mensual['tasa_pct']=mensual['tasa']*100
display(mensual)
plt.figure(figsize=(12,5));sns.lineplot(data=mensual,x='month',y='tasa_pct',hue='year',marker='o');plt.xticks(range(1,13));plt.title('Retrasos por mes (muestra)');plt.ylabel('% >=15 min');plt.show()
print('ADVERTENCIA: solo dos años y muestra de 5,000 por mes; verificar en Gold completo.')

## 6. Horario, aeropuertos y aerolíneas

In [ ]:
por_hora=eda.groupby('scheduled_hour')['delay_15'].agg(['size','mean']).reset_index()
por_hora['tasa_pct']=por_hora['mean']*100
display(por_hora)
fig,ax=plt.subplots(2,1,figsize=(12,7),sharex=True)
ax[0].bar(por_hora['scheduled_hour'],por_hora['size']);ax[0].set_ylabel('Vuelos')
ax[1].plot(por_hora['scheduled_hour'],por_hora['tasa_pct'],marker='o');ax[1].set_ylabel('% retrasados');ax[1].set_xlabel('Hora local programada')
plt.tight_layout();plt.show()
for campo in ['Origin','Reporting_Airline']:
    resumen=eda.groupby(campo)['delay_15'].agg(vuelos='size',tasa='mean').sort_values('vuelos',ascending=False).head(15)
    display(resumen.assign(tasa_pct=100*resumen['tasa']))

## 7. Evolución diaria

In [ ]:
diario=eda.groupby('FlightDate').agg(vuelos=('delay_15','size'), tasa=('delay_15','mean')).reset_index().sort_values('FlightDate')
fig,ax=plt.subplots(2,1,figsize=(13,7),sharex=True)
ax[0].bar(diario['FlightDate'],diario['vuelos']);ax[0].set_ylabel('Vuelos')
ax[1].plot(diario['FlightDate'],diario['tasa']*100);ax[1].set_ylabel('% retrasados')
plt.tight_layout();plt.show()

## 8. Lectura de resultados y siguiente etapa

**Este notebook es para EDA preliminar sobre una muestra.** El análisis formal deberá ejecutarse en Databricks con tablas Gold construidas desde Bronze/Silver, usando los vuelos completos y una política de validación reproducible.

No se ha hecho aquí el ajuste de distribuciones, GARCH, Monte Carlo ni el entrenamiento final de XGBoost a tres horas. Nunca presentar resultados generados por un notebook nuevo como si fueran los resultados antiguos.